# Phase P1: Setup, Hardware Preflight & Dataset Preparation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/00_setup_preflight.ipynb)


In [ ]:
import os, sys, json, subprocess
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/thinkTinkerGreen/train_with_reversibility.git"
REPO_REF = "main"

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/revllm/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/runs', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/probes', exist_ok=True)

    repo_dir = "/content/train_with_reversibility"
    if not os.path.exists(repo_dir):
        print(f"Cloning {REPO_URL} into {repo_dir}...")
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    else:
        print(f"Repository already present at {repo_dir}. Fetching updates...")
        subprocess.run(["git", "-C", repo_dir, "fetch", "--all"], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{REPO_REF}"], check=False)

    os.chdir(repo_dir)

    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    src_dir = os.path.join(repo_dir, "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

print(f"Current working directory: {os.getcwd()}")
print(f"Execution mode: {'Smoke (CPU/CI)' if SMOKE else 'Colab GPU'}")


In [ ]:
# Hardware and environment verification
import torch

env_data = {
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "None",
    "python_version": sys.version.split()[0],
}

if not torch.cuda.is_available() and not SMOKE:
    raise RuntimeError("No GPU detected! Ensure Colab runtime is set to GPU (T4).")

os.makedirs("results", exist_ok=True)
with open("results/env.json", "w") as f:
    json.dump(env_data, f, indent=2)
print("Hardware environment recorded:", json.dumps(env_data, indent=2))

if IN_COLAB and not SMOKE:
    import shutil
    shutil.copy2("results/env.json", "/content/drive/MyDrive/revllm/env.json")


In [ ]:
# Install dependencies without touching pre-installed PyTorch
if not SMOKE and IN_COLAB:
    print("Installing build and runtime dependencies...")
    deps = ["hatchling", "pyyaml", "tokenizers", "huggingface-hub", "datasets", "tqdm"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
    print("Package revllm successfully installed!")

# Run preflight verification script
from scripts.preflight import run_preflight
preflight_results = run_preflight()
print("Preflight verification status:", preflight_results.get("arch_checks"))

if IN_COLAB and not SMOKE:
    import shutil
    shutil.copy2("results/preflight.json", "/content/drive/MyDrive/revllm/preflight.json")


In [ ]:
# Prepare dataset and tokenizer
from scripts.build_data import main as build_data_main
# In smoke mode, creates tiny dataset; in full Colab mode builds full FineWeb-Edu split
build_data_main([])
if IN_COLAB and not SMOKE:
    import shutil
    shutil.copy2("results/data_manifest.json", "/content/drive/MyDrive/revllm/data/data_manifest.json")
    if os.path.exists("data/tokenizer.json"):
        shutil.copy2("data/tokenizer.json", "/content/drive/MyDrive/revllm/data/tokenizer.json")
    if os.path.exists("data/train.bin"):
        shutil.copy2("data/train.bin", "/content/drive/MyDrive/revllm/data/train.bin")
    if os.path.exists("data/val.bin"):
        shutil.copy2("data/val.bin", "/content/drive/MyDrive/revllm/data/val.bin")
print("Dataset preparation complete.")
